# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

repo_root = Path.cwd().resolve()
for candidate in [repo_root, *repo_root.parents]:
    cache_file = candidate / 'work' / 'outputs' / 'february_march_features.parquet'
    if cache_file.exists():
        repo_root = candidate
        break

cache_path = repo_root / 'work' / 'outputs' / 'february_march_features.parquet'
dataframe = pd.read_parquet(cache_path)
dataframe = dataframe[dataframe['future_decline_label'].notna()].copy()

# Same feature prep as Week 5
X = dataframe[['prior_impressions', 'prior_clicks', 'prior_avg_position',
               'prior_sessions', 'prior_engagement_rate']].copy()
X['prior_avg_position'] = X['prior_avg_position'].fillna(999)
X['prior_sessions'] = X['prior_sessions'].fillna(0)
X['prior_engagement_rate'] = X['prior_engagement_rate'].fillna(0)
X['prior_ctr'] = (dataframe['prior_clicks'] / dataframe['prior_impressions'].replace(0, np.nan)).fillna(0)

model_features = ['prior_impressions', 'prior_clicks', 'prior_ctr',
                   'prior_avg_position', 'prior_sessions', 'prior_engagement_rate']
X = X[model_features]
y = dataframe['future_decline_label'].values
groups = dataframe['client_hash_id']

def precision_at_20(y_true, scores):
    order = np.argsort(-scores)
    return y_true[order][:20].mean()

def fit_and_score(X_train, X_test, y_train, y_test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    model = LogisticRegression(random_state=42, max_iter=1000, solver='lbfgs')
    model.fit(X_train_scaled, y_train)
    scores = model.predict_proba(X_test_scaled)[:, 1]
    return {
        'test_rows': len(y_test),
        'base_rate': float(y_test.mean()),
        'precision_at_20': precision_at_20(y_test, scores),
    }

# BEFORE: current approach — fit AND evaluate on all rows (in-sample, no held-out test)
before_result = fit_and_score(X, X, y, y)

# AFTER: honest split — grouped by client so no client appears on both sides
splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
after_result = fit_and_score(X.iloc[train_idx], X.iloc[test_idx], y[train_idx], y[test_idx])

overlap = set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])
print(f"Client overlap in the honest split: {len(overlap)} (should be 0)")

comparison = pd.DataFrame([
    {'split': 'before_in_sample_no_split', **before_result},
    {'split': 'after_grouped_by_client', **after_result},
])
comparison['gap_vs_before'] = comparison['precision_at_20'] - before_result['precision_at_20']
comparison

Client overlap in the honest split: 0 (should be 0)


,split,test_rows,base_rate,precision_at_20,gap_vs_before
0,before_in_sample_no_split,80322,0.217549,0.45,0.0
1,after_grouped_by_client,28904,0.187690,0.35,-0.1


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.